In [60]:
import pickle
from pathlib import Path
import os
import numpy as np
import pandas as pd
from IPython.display import display


# Paste the absolute or relative path to your specific experiment folder here
# Using ".." to go up one level from 'notebooks/' to the project root
EXPERIMENT_FOLDER = Path("../data/processed/CF1_delta06/") 

# Load the pickle files
data = {}
for pkl_name in ["standalone_values.pkl", "characteristic_function.pkl", "allocation_results.pkl"]:
    file_path = EXPERIMENT_FOLDER / pkl_name
    # Resolve to absolute path to verify exactly where it is looking
    if file_path.resolve().exists():
        with open(file_path, "rb") as f:
            data[pkl_name] = pickle.load(f)
    else:
        print(f"Warning: {pkl_name} not found at {file_path.resolve()}")

In [61]:
summary = {"Experiment": EXPERIMENT_FOLDER.name}

# Stage 02: Solo Optimisation Summary
if "standalone_values.pkl" in data:
    sv = data["standalone_values.pkl"]
    statuses = [r["status"] for r in sv.values()]
    svs = [r["standalone_value"] for r in sv.values()]
    
    summary["N_optimal_solo"] = sum(1 for s in statuses if s == "optimal")
    summary["N_profitable_solo"] = sum(1 for r in sv.values() if r["raw_obj_value"] > 0)
    summary["Total_Solo_Value_INR"] = round(sum(svs))

# Stage 03: Coalition Enumeration Summary
if "characteristic_function.pkl" in data:
    cf = data["characteristic_function.pkl"]
    summary["v_Grand_INR"] = round(cf.get("grand_coalition_value", 0))
    
    all_vals = np.array(list(cf.get("characteristic_function", {}).values()))
    summary["N_Positive_Subcoalitions"] = (all_vals > 0).sum()

# Stage 04: Allocation and Stability Summary
if "allocation_results.pkl" in data:
    alloc = data["allocation_results.pkl"]
    lc_meta = alloc.get("least_core_meta", {})
    verif = alloc.get("verification", {})
    
    summary["Coop_Gain_INR"] = round(alloc.get("v_grand", 0) - sum(svs)) if "standalone_values.pkl" in data else np.nan
    summary["Core_Nonempty"] = lc_meta.get("core_nonempty")
    summary["LC_Epsilon_Star"] = round(lc_meta.get("epsilon_star", float('nan')), 2)
    summary["Shapley_Violations"] = verif.get("Shapley", {}).get("n_core_violations")
    summary["Equal_Split_Violations"] = verif.get("Equal Split", {}).get("n_core_violations")

df_summary = pd.DataFrame([summary])
display(df_summary)

,Experiment,N_optimal_solo,N_profitable_solo,Total_Solo_Value_INR,v_Grand_INR,N_Positive_Subcoalitions,Coop_Gain_INR,Core_Nonempty,LC_Epsilon_Star,Shapley_Violations,Equal_Split_Violations
0,CF1_delta06,0,0,0,1229884,13415,1229884,True,0.0,2684,9974


In [62]:
if "allocation_results.pkl" in data:
    res = data["allocation_results.pkl"]
    farmer_ids = res["farmer_ids"]
    sv_vec = res["standalone_values"]
    allocs = res.get("allocations", {})
    
    # Extract mechanisms safely
    lc = allocs.get("least_core", [np.nan]*len(farmer_ids))
    nuc = allocs.get("nucleolus", [np.nan]*len(farmer_ids))
    shp = allocs.get("shapley", [np.nan]*len(farmer_ids))
    es = allocs.get("equal_split", [np.nan]*len(farmer_ids))
    ps = allocs.get("proportional", [np.nan]*len(farmer_ids))

    rows = []
    for i, fid in enumerate(farmer_ids):
        rows.append({
            "Farmer_ID": fid,
            "Standalone_Value": round(sv_vec[i], 2),
            "Least_Core": round(lc[i], 2),
            "Nucleolus": round(nuc[i], 2),
            "Shapley": round(shp[i], 2),
            "Equal_Split": round(es[i], 2),
            "Proportional": round(ps[i], 2)
        })
        
    df_allocations = pd.DataFrame(rows)
    
    # Add a totals row
    totals = df_allocations.sum(numeric_only=True)
    totals["Farmer_ID"] = "TOTAL"
    df_allocations.loc[len(df_allocations)] = totals
    
    display(df_allocations)
else:
    print("allocation_results.pkl not loaded. Cannot display per-farmer allocations.")

,Farmer_ID,Standalone_Value,Least_Core,Nucleolus,Shapley,Equal_Split,Proportional
0,F0001,0.0,76867.73,75954.75,84920.29,81992.24,76056.15
1,F0002,0.0,19243.80,18370.61,7012.23,81992.24,19477.79
2,F0003,0.0,120306.85,119371.96,149278.46,81992.24,118721.79
3,F0004,0.0,145665.64,144853.07,140349.89,81992.24,143764.67
4,F0005,0.0,64553.27,63683.58,57027.80,81992.24,63998.47
5,F0006,0.0,40958.71,40083.85,31853.08,81992.24,40810.62
6,F0007,0.0,56020.78,55187.87,46883.98,81992.24,55650.84
7,F0008,0.0,23983.32,23091.00,9468.32,81992.24,24115.36
8,F0009,0.0,137196.81,136359.61,134378.74,81992.24,135417.04
9,F0010,0.0,123990.75,123147.08,124633.82,81992.24,122431.85


In [63]:
pkl_path = EXPERIMENT_FOLDER / "allocation_results.pkl"

if pkl_path.exists():
    with open(pkl_path, "rb") as f:
        res = pickle.load(f)

    # Extract core variables
    v_gc = res.get("v_grand", np.nan)
    sv_vec = res.get("standalone_values", np.array([]))
    lc_meta = res.get("least_core_meta", {})
    verif = res.get("verification", {})

    # Helper for extracting max excess safely
    def max_excess(name):
        v = verif.get(name, {}).get("max_excess", float("nan"))
        return round(v, 2) if isinstance(v, float) else np.nan

    # 2. Build the exact row you requested
    summary = {
        "Experiment": EXPERIMENT_FOLDER.name,
        "v_Grand_INR": round(v_gc) if not np.isnan(v_gc) else np.nan,
        "Coop_Gain_INR": round(v_gc - sv_vec.sum()) if len(sv_vec) > 0 else np.nan,
        "Core_nonempty": lc_meta.get("core_nonempty", False),
        "LC_slack_eps": round(lc_meta.get("epsilon_star", float("nan")), 2),
        "Shap_viol": verif.get("Shapley", {}).get("n_core_violations", "N/A"),
        "Shap_max_excess": max_excess("Shapley"),
        "ES_viol": verif.get("Equal Split", {}).get("n_core_violations", "N/A"),
    }

    # 3. Display as a Pandas DataFrame
    df_stage04 = pd.DataFrame([summary])
    display(df_stage04)
    
    # Optional: If you want the LaTeX code directly for your paper:
    # print(df_stage04.to_latex(index=False))

else:
    print(f"File not found: {pkl_path.resolve()}")

,Experiment,v_Grand_INR,Coop_Gain_INR,Core_nonempty,LC_slack_eps,Shap_viol,Shap_max_excess,ES_viol
0,CF1_delta06,1229884,1229884,True,0.0,2684,99396.99,9974
